# Hyperparameter Tuning for Phishing URL Detection

## 1. Introduction

Hyperparameter tuning is the process of finding suitable values for the parameters that control the behavior of a machine learning algorithm.

In the previous notebook, baseline classification models were trained using their initial configurations. Hyperparameter tuning will now be used to systematically test different configurations and identify models that provide improved validation performance.

## 2. Objectives

The main objectives of this notebook are:

- Load the selected feature dataset.
- Recreate the leakage-free URL-group train-test split.
- Apply feature scaling using training data only.
- Define appropriate hyperparameter search spaces.
- Tune the baseline machine learning models.
- Use cross-validation during the search process.
- Record the best hyperparameters obtained for each tuned model.
- Compare baseline and tuned performance.
- Save the tuned models for the final evaluation stage.

## 3. Models for Tuning

The following models will be considered:

1. Logistic Regression
2. Decision Tree
3. Random Forest

Support Vector Machine will not be included in the initial tuning search because training and repeated cross-validation on the full dataset can be computationally expensive.

## 4. Hyperparameter Search

Grid Search with stratified cross-validation will be used for the selected models.

The search will evaluate predefined combinations of hyperparameters and select the configuration that produces the best cross-validation score.

## 5. Evaluation Metric

F1-score will be used as the primary tuning metric because it considers both precision and recall.

ROC-AUC and the other classification metrics will be examined after tuning during the model evaluation stage.

## 6. Leakage Prevention

The test dataset will remain completely unseen during hyperparameter tuning.

All hyperparameter decisions will be based only on the training data and cross-validation results.

The URL-group splitting strategy will continue to ensure that the same URL does not occur in both training and testing datasets.

## 7. Expected Outcome

At the end of this notebook, tuned versions of the selected baseline models will be available along with:

- Best hyperparameters
- Cross-validation scores
- Baseline performance
- Tuned performance
- Saved trained models

These tuned models will be used in the subsequent model evaluation and comparison stage.

In [1]:
# Import libraries and load the selected feature dataset

import pandas as pd
import numpy as np
import os
import joblib
import time

from sklearn.model_selection import train_test_split

dataset_path = "../dataset/PhiUSIIL_Selected_Features.csv"

df = pd.read_csv(dataset_path)

print("HYPERPARAMETER TUNING ENVIRONMENT READY")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

HYPERPARAMETER TUNING ENVIRONMENT READY
Dataset shape: (235795, 41)
Number of records: 235795
Number of columns: 41


In [2]:
# Separate predictive features and target variable

target_column = "label"

X = df.drop(columns=[target_column])
y = df[target_column]

print("FEATURE AND TARGET PREPARATION")
print("=" * 60)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Number of predictive features:", X.shape[1])

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget mapping:")
print("0 = Phishing")
print("1 = Legitimate")

FEATURE AND TARGET PREPARATION
Feature matrix shape: (235795, 40)
Target vector shape: (235795,)
Number of predictive features: 40

Target distribution:
label
0    100945
1    134850
Name: count, dtype: int64

Target mapping:
0 = Phishing
1 = Legitimate


In [3]:
# Create a leakage-free URL-group train/test split

print("URL-GROUP TRAIN-TEST SPLIT")
print("=" * 60)

# Load URL information once
url_data = pd.read_csv(
    "../dataset/PhiUSIIL_Engineered_Features.csv",
    usecols=["URL", "label"]
)

# Create one record per unique URL
url_groups = url_data.drop_duplicates(subset="URL")

# Split unique URLs
train_url_groups, test_url_groups = train_test_split(
    url_groups,
    test_size=0.20,
    random_state=42,
    stratify=url_groups["label"]
)

# Create URL sets
train_url_set = set(train_url_groups["URL"])
test_url_set = set(test_url_groups["URL"])

# Assign records to train/test groups
train_mask = url_data["URL"].isin(train_url_set)
test_mask = url_data["URL"].isin(test_url_set)

# Create training and testing datasets
X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

# Verify no URL appears in both sets
common_urls = train_url_set.intersection(test_url_set)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining percentage:",
      round(len(X_train) / len(df) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(df) * 100, 2), "%")

print("\nURLs appearing in both sets:", len(common_urls))

if len(common_urls) == 0:
    print("✅ No URL leakage detected.")
else:
    print("❌ URL leakage detected.")

URL-GROUP TRAIN-TEST SPLIT
Training records: 188644
Testing records: 47151

Training percentage: 80.0 %
Testing percentage: 20.0 %

URLs appearing in both sets: 0
✅ No URL leakage detected.


In [4]:
# Feature scaling using training data only

from sklearn.preprocessing import StandardScaler

print("FEATURE SCALING")
print("=" * 60)

scaler = StandardScaler()

# Fit the scaler only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the training-fitted scaler
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nTraining scaled mean:",
      round(X_train_scaled.mean().mean(), 4))

print("Training scaled standard deviation:",
      round(X_train_scaled.std().mean(), 4))

print("\n✅ Feature scaling completed.")
print("Scaler fitted only on training data.")

FEATURE SCALING
Training data shape: (188644, 40)
Testing data shape: (47151, 40)

Training scaled mean: 0.0
Training scaled standard deviation: 1.0

✅ Feature scaling completed.
Scaler fitted only on training data.


In [5]:
# Define models and hyperparameter search spaces

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

models_for_tuning = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
}

param_grids = {

    "Logistic Regression": {
        "C": [0.1, 1, 10],
        "solver": ["liblinear"]
    },

    "Decision Tree": {
        "max_depth": [5, 10, 20, None],
        "min_samples_split": [2, 10],
        "min_samples_leaf": [1, 5]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 5]
    }
}

for model_name in models_for_tuning:
    combinations = 1

    for values in param_grids[model_name].values():
        combinations *= len(values)

    print(f"\n{model_name}")
    print("Hyperparameter combinations:", combinations)

print("\n✅ Hyperparameter search spaces defined.")

HYPERPARAMETER SEARCH SPACE

Logistic Regression
Hyperparameter combinations: 3

Decision Tree
Hyperparameter combinations: 16

Random Forest
Hyperparameter combinations: 12

✅ Hyperparameter search spaces defined.


In [6]:
# Perform hyperparameter tuning using GridSearchCV

from sklearn.model_selection import GridSearchCV, StratifiedKFold

print("HYPERPARAMETER TUNING")
print("=" * 60)

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

tuned_models = {}
tuning_results = {}
best_params = {}

for model_name in models_for_tuning:

    print(f"\nTuning {model_name}...")
    print("-" * 50)

    start_time = time.time()

    grid_search = GridSearchCV(
        estimator=models_for_tuning[model_name],
        param_grid=param_grids[model_name],
        scoring="f1",
        cv=cv_strategy,
        n_jobs=-1,
        verbose=1
    )

    # Use scaled data for all models
    grid_search.fit(X_train_scaled, y_train)

    elapsed_time = time.time() - start_time

    tuned_models[model_name] = grid_search.best_estimator_
    tuning_results[model_name] = grid_search.best_score_
    best_params[model_name] = grid_search.best_params_

    print(f"\n✅ {model_name} tuning completed.")
    print("Best F1-score:", round(grid_search.best_score_, 4))
    print("Best parameters:", grid_search.best_params_)
    print("Tuning time:", round(elapsed_time, 2), "seconds")

print("\n" + "=" * 60)
print("ALL HYPERPARAMETER TUNING COMPLETED")

HYPERPARAMETER TUNING

Tuning Logistic Regression...
--------------------------------------------------
Fitting 5 folds for each of 3 candidates, totalling 15 fits

✅ Logistic Regression tuning completed.
Best F1-score: 0.9999
Best parameters: {'C': 10, 'solver': 'liblinear'}
Tuning time: 36.41 seconds

Tuning Decision Tree...
--------------------------------------------------
Fitting 5 folds for each of 16 candidates, totalling 80 fits

✅ Decision Tree tuning completed.
Best F1-score: 1.0
Best parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
Tuning time: 16.51 seconds

Tuning Random Forest...
--------------------------------------------------
Fitting 5 folds for each of 12 candidates, totalling 60 fits


KeyboardInterrupt: 

In [7]:
# Evaluate tuned models on the test dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("TUNED MODEL TEST EVALUATION")
print("=" * 60)

tuned_predictions = {}
tuned_probabilities = {}
tuned_results = []

for model_name, model in tuned_models.items():

    print(f"\nEvaluating {model_name}...")

    # Predictions
    y_pred = model.predict(X_test_scaled)

    # Probability predictions
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Store predictions
    tuned_predictions[model_name] = y_pred
    tuned_probabilities[model_name] = y_prob

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    tuned_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

    print(f"✅ {model_name} evaluated.")

tuned_results_df = pd.DataFrame(tuned_results)

print("\n" + "=" * 60)
print("TUNED MODEL RESULTS")

display(tuned_results_df.round(4))

TUNED MODEL TEST EVALUATION

Evaluating Logistic Regression...
✅ Logistic Regression evaluated.

Evaluating Decision Tree...
✅ Decision Tree evaluated.

TUNED MODEL RESULTS


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,1.0,0.9999,1.0,1.0,1.0
1,Decision Tree,1.0,1.0000,1.0,1.0,1.0


In [9]:
# Compare baseline vs tuned model performance

print("BASELINE VS TUNED MODEL COMPARISON")
print("=" * 60)

# Load baseline results saved from Notebook 06
baseline_results_path = "../results/baseline_model_results.csv"

baseline_results_df = pd.read_csv(baseline_results_path)

comparison_rows = []

for model_name in tuned_models.keys():

    # Baseline results
    baseline_row = baseline_results_df[
        baseline_results_df["Model"] == model_name
    ].iloc[0]

    # Tuned results
    tuned_row = tuned_results_df[
        tuned_results_df["Model"] == model_name
    ].iloc[0]

    comparison_rows.append({
        "Model": model_name,

        "Baseline Accuracy": baseline_row["Accuracy"],
        "Tuned Accuracy": tuned_row["Accuracy"],

        "Baseline Precision": baseline_row["Precision"],
        "Tuned Precision": tuned_row["Precision"],

        "Baseline Recall": baseline_row["Recall"],
        "Tuned Recall": tuned_row["Recall"],

        "Baseline F1": baseline_row["F1-Score"],
        "Tuned F1": tuned_row["F1-Score"],

        "Baseline ROC-AUC": baseline_row["ROC-AUC"],
        "Tuned ROC-AUC": tuned_row["ROC-AUC"]
    })

baseline_tuned_comparison = pd.DataFrame(comparison_rows)

display(
    baseline_tuned_comparison.round(4)
)

print("\n✅ Baseline and tuned performance comparison completed.")

BASELINE VS TUNED MODEL COMPARISON


,Model,Baseline Accuracy,Tuned Accuracy,Baseline Precision,Tuned Precision,Baseline Recall,Tuned Recall,Baseline F1,Tuned F1,Baseline ROC-AUC,Tuned ROC-AUC
0,Logistic Regression,0.9999,1.0,0.9999,0.9999,1.0,1.0,0.9999,1.0,1.0,1.0
1,Decision Tree,1.0000,1.0,1.0000,1.0000,1.0,1.0,1.0000,1.0,1.0,1.0



✅ Baseline and tuned performance comparison completed.


In [10]:
# Display the best hyperparameters found during tuning

print("BEST HYPERPARAMETERS")
print("=" * 60)

best_parameters_df = pd.DataFrame([
    {
        "Model": model_name,
        "Best Parameters": best_params[model_name],
        "Best CV F1-Score": tuning_results[model_name]
    }
    for model_name in tuned_models.keys()
])

display(
    best_parameters_df.round(4)
)

print("\n✅ Best hyperparameters recorded.")

BEST HYPERPARAMETERS


,Model,Best Parameters,Best CV F1-Score
0,Logistic Regression,"{'C': 10, 'solver': 'liblinear'}",0.9999
1,Decision Tree,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s...",1.0000



✅ Best hyperparameters recorded.


In [11]:
# Save hyperparameter tuning results

print("SAVING HYPERPARAMETER TUNING RESULTS")
print("=" * 60)

# Create results directory
os.makedirs("../results", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# Save tuned model performance
tuned_results_path = "../results/tuned_model_results.csv"

tuned_results_df.to_csv(
    tuned_results_path,
    index=False
)

# Save baseline vs tuned comparison
comparison_path = "../results/baseline_vs_tuned_comparison.csv"

baseline_tuned_comparison.to_csv(
    comparison_path,
    index=False
)

# Save best hyperparameters
best_parameters_path = "../results/best_hyperparameters.csv"

best_parameters_df.to_csv(
    best_parameters_path,
    index=False
)

print("✅ Tuned model results saved to:")
print(tuned_results_path)

print("\n✅ Baseline vs tuned comparison saved to:")
print(comparison_path)

print("\n✅ Best hyperparameters saved to:")
print(best_parameters_path)

SAVING HYPERPARAMETER TUNING RESULTS
✅ Tuned model results saved to:
../results/tuned_model_results.csv

✅ Baseline vs tuned comparison saved to:
../results/baseline_vs_tuned_comparison.csv

✅ Best hyperparameters saved to:
../results/best_hyperparameters.csv


In [12]:
# Save tuned machine learning models

print("SAVING TUNED MODELS")
print("=" * 60)

for model_name, model in tuned_models.items():

    file_name = (
        model_name.lower()
        .replace(" ", "_")
        + "_tuned.pkl"
    )

    model_path = os.path.join(
        "../models",
        file_name
    )

    joblib.dump(model, model_path)

    print(f"✅ Saved: {model_path}")

# Save the scaler used during tuning
scaler_path = "../models/tuning_feature_scaler.pkl"

joblib.dump(
    scaler,
    scaler_path
)

print(f"✅ Saved: {scaler_path}")

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING COMPLETED")
print("Tuned models are ready for final evaluation.")

SAVING TUNED MODELS
✅ Saved: ../models\logistic_regression_tuned.pkl
✅ Saved: ../models\decision_tree_tuned.pkl
✅ Saved: ../models/tuning_feature_scaler.pkl

HYPERPARAMETER TUNING COMPLETED
Tuned models are ready for final evaluation.
